
# Week 10 · Class 5 — Independent Capstone
## Historical Production Data Quality Pipeline

**Assessment notebook — not a guided teaching lesson.** Produce a self-contained, reproducible analytic report. Include your own explanations, assumptions, code outputs, and findings. This notebook must call and demonstrate the functions you implement in `src/production_pipeline.py`.

**Important:** Use the full project brief as the source of truth. Do not delete the original dataset or change input values.

.## 1. Problem statement and objectives
Write 1–2 paragraphs describing why missing 6-hour observations, duplicates, incorrect measurements, and changing sensor reliability matter. State the required outputs and your policy for zeros versus missing readings.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from src.production_pipeline import ingest_and_validate, repair_and_align, summarize_daily, run_pipeline
INPUT = Path("data/historical_production_capstone.csv")
OUTPUT = Path("outputs")
print("Input present:", INPUT.exists())

## 2. Data ingestion and schema validation
Show raw row count, column names, data types, sample rows and invalid timestamps. Explain why timestamp parsing uses `errors="coerce"`.

In [ ]:
# TODO: inspect the raw CSV and demonstrate ingest_and_validate(INPUT)

# Section 2: Data Ingestion and Schema Validation

from pathlib import Path
import pandas as pd

# Locate the original dataset
INPUT = Path("../data/historical_production_capstone.csv")

# Load the raw dataset
raw_df = pd.read_csv(INPUT)

# Display raw row count
print("RAW ROW COUNT:", len(raw_df))

# Display column names
print("\nCOLUMN NAMES:")
print(raw_df.columns.tolist())

# Display data types
print("\nDATA TYPES:")
print(raw_df.dtypes)

# Display first five rows
print("\nSAMPLE ROWS:")
display(raw_df.head())


## 3. Deduplication, alignment, and cleaning
Show how latest-ingestion duplicate precedence works. Explain how missing *timestamps* differ from missing *values*. Display a before-and-after example for a chosen well. Explain how you prevent false oil production totals.

In [ ]:
# TODO: apply repair_and_align; show per-well sample rows, missingness counts, and quality events

# Check timestamp validity

raw_df["timestamp_parsed"] = pd.to_datetime(
    raw_df["timestamp"],
    errors="coerce"
)

invalid_timestamps = raw_df[
    raw_df["timestamp"].notna()
    & raw_df["timestamp_parsed"].isna()
]

print("INVALID TIMESTAMPS:", len(invalid_timestamps))

display(invalid_timestamps.head(10))


## 4. Daily resampling / aggregation
Report daily oil, water and gas totals, observation completeness, and incomplete days. Explain why sum(min_count=1) protects all-missing groups.

In [ ]:
# TODO: call summarize_daily and display a well/day summary

# SECTION 4: DAILY RESAMPLING / AGGREGATION

from src.production_pipeline import summarize_daily

# 1. Generate daily production summaries
daily_summary = summarize_daily(cleaned_rows)

# 2. Display total daily summary records
print("TOTAL DAILY SUMMARY ROWS:", len(daily_summary))

# 3. Display daily oil, water and gas production
print("\nDAILY PRODUCTION TOTALS:")

display(
    daily_summary[
        [
            "well_id",
            "date",
            "daily_oil_bbl",
            "daily_water_bbl",
            "daily_gas_mscf"
        ]
    ]
)

# 4. Display observation completeness
print("\nOBSERVATION COMPLETENESS:")

display(
    daily_summary[
        [
            "well_id",
            "date",
            "oil_bbl_valid_readings",
            "water_bbl_valid_readings",
            "gas_mscf_valid_readings",
            "is_complete"
        ]
    ]
)

# 5. Identify incomplete production days
incomplete_days = daily_summary[
    ~daily_summary["is_complete"]
]

print("\nNUMBER OF INCOMPLETE DAYS:", len(incomplete_days))

print("\nINCOMPLETE DAY DETAILS:")

display(incomplete_days)

# 6. Check overall completeness
print("\nCOMPLETE DAYS:", daily_summary["is_complete"].sum())

print("INCOMPLETE DAYS:", (~daily_summary["is_complete"]).sum())

# 7. Display daily summary data types
print("\nDAILY SUMMARY DATA TYPES:")
print(daily_summary.dtypes)


## 5. Validation evidence and interpretation
Provide results of tests you wrote, expected-vs-actual checks for row counts, and at least three operational insights or limitations. Note that the data are synthetic.

In [ ]:

# SECTION 5: VALIDATION EVIDENCE AND INTERPRETATION
# 1. Run the project tests
print("RUNNING PIPELINE TESTS...")

test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-v"],
    capture_output=True,
    text=True
)

print(test_result.stdout)

if test_result.stderr:
    print(test_result.stderr)

print("TEST EXIT CODE:", test_result.returncode)

if test_result.returncode == 0:
    print("RESULT: ALL TESTS PASSED")
else:
    print("RESULT: SOME TESTS FAILED - REVIEW THE OUTPUT")

# 2. Expected versus actual row counts
raw_count = len(pd.read_csv(INPUT))

valid_count = len(valid_rows)
rejected_count = len(rejected_rows)

actual_source_count = valid_count + rejected_count

print("\nROW COUNT VALIDATION")
print("--------------------")
print("Expected raw rows:", raw_count)
print("Valid rows:", valid_count)
print("Rejected rows:", rejected_count)
print("Actual reconciled rows:", actual_source_count)

print(
    "Row count reconciliation:",
    "PASSED" if raw_count == actual_source_count else "FAILED"
)

# 3. Compare rows before and after cleaning
print("\nCLEANING VALIDATION")
print("-------------------")
print("Valid source rows:", len(valid_rows))
print("Rows after alignment:", len(cleaned_rows))
print("Daily summary rows:", len(daily_summary))

# 4. Run the complete pipeline
print("\nRUNNING COMPLETE PIPELINE...")

run_pipeline(INPUT, OUTPUT)

# 5. Display the four generated output files
print("\nGENERATED OUTPUT FILES:")

output_files = sorted(
    file.name for file in OUTPUT.iterdir()
    if file.is_file()
)

for file_name in output_files:
    print("-", file_name)

print("\nTOTAL OUTPUT FILES:", len(output_files))


## 6. Conclusions and reproducibility
Explain your cleaning decisions, their impact on interpretation, and how another analyst can rerun the pipeline. Include a link to the GitHub Pull Request in your README (do not hard-code a private token).

## 6. Conclusions and Reproducibility

### Conclusion

This project developed a historical production data quality pipeline for processing synthetic oil and gas production records.

The pipeline addresses common data quality problems, including duplicate observations, invalid timestamps, missing six-hour readings and unreliable sensor measurements.

The following cleaning decisions were applied:

1. **Data validation:** Invalid records were identified during ingestion to prevent unreliable data from entering the analysis.
2. **Deduplication:** Where duplicate observations existed, the latest-ingested record was retained.
3. **Timestamp alignment:** Production records were aligned to the expected six-hour observation intervals.
4. **Missing measurements:** Missing production values were preserved as missing rather than automatically replaced with zero.
5. **Daily aggregation:** Six-hour observations were summarized into daily oil, water and gas production totals.
6. **Completeness checks:** Daily observation completeness was calculated to help identify days with insufficient measurements.

These decisions improve the reliability of production reporting and reduce the risk of misleading production totals. However, missing measurements and incomplete observations remain important limitations when interpreting the results.

### Reproducibility

Another analyst can reproduce this project by following these steps:

1. Clone the project repository from GitHub.
2. Navigate to the project directory.
3. Create and activate a Python virtual environment.
4. Install the required packages from `requirements.txt`.
5. Ensure the original dataset is available in the `data` directory.
6. Run the automated tests using `pytest -v`.
7. Open the Jupyter Notebook and execute its cells in order.
8. Run the production pipeline to regenerate the output files.

The pipeline is implemented in `src/production_pipeline.py`, while the notebook demonstrates its functions and presents the analysis results.

The original dataset is preserved, and generated reports are saved in the outputs directory.

### Final Note

This project uses synthetic historical production data. The results demonstrate a reproducible data quality workflow but should not be interpreted as actual oilfield production measurements.
